# **Hospital Healthcare Data Analysis with SQL and Power BI**
by Canly Hansen Sudirman

## 1. Context and Data Understanding

This dataset is found in Kaggle (https://www.kaggle.com/datasets/prasad22/healthcare-dataset). This project is initiated because I feel the need to showcase my ability in SQL and Power BI. I like that it is a neat and compact project. I chose hospital healthcare for this project as some sort of shout-out to pharmacy, the field of study to which I got my degree on, nothing more conclusive than that. I feel like lately, a lot of writings have been checked by AI and although I use it here and there, to apply it in every occasion is not wise either to prevent myself from being lost in touch, so all of this writing and code is free from AI. I understand that this project is far from perfect, but just like every other expert, some of the first steps will be uneasy, but the most important thing is that we start anyway.

Now, it is said that this synthetic healthcare dataset has been created to serve as a valuable resource for data science, machine learning, and data analysis enthusiasts. It is designed to mimic real-world healthcare data, enabling users to practice, develop, and showcase their data manipulation and analysis skills in the context of the healthcare industry, so I would not give too much thought about the data itself and would pinpoint the query (SQL) and the visualization itself (Power BI), although at some cases, the discussion of the origin of the data shall not be separated.

Here, I am looking at every column in the dataset. Name is a bit messsed up, but that's not a big concern. We are here trying to find trend and patterns hidden in the dataset, so I would like to lay some grounds for correlation. But before doing all that, let's clean this data first.

## 2. Data Cleaning

There are various things that I wanted to do here. First, I checked any data type by using DESC statement. I found out that the dates are in incorrect type so I changed them. 
```
ALTER TABLE health MODIFY `Date of Admission` DATE;
ALTER TABLE health MODIFY `Discharge Date` DATE;
```

Next, I'd like to start with checking feature relevance. But since it is quite simple, there is not much to see in here. There are some information that I believe could be useful in the visualization, but if I have to delete columns to prevent it from being bloated, I would delete Name and Room Number.

### 2.1. Missing value check

The missing value check can be performed using this query:

```
SELECT * FROM health 
WHERE Age = '' OR Gender = '' OR `Blood Type` = '' OR `Medical Condition` = '' OR `Date of Admission` = '' OR `Insurance Provider` = '' OR `Billing Amount` = '' OR `Discharge Date` = '';
```

There are no missing values in the dataset, so it is safe to move forward.

### 2.2. Duplicate check

The duplicate check can be performed using this query:

```
SELECT Name, COUNT(*) AS occurrences
FROM health
GROUP BY Name
HAVING COUNT(*) > 1;
```

There are duplicates in the dataset, and I planned to delete them using CTE, but it turns out that it does not work because Error Code 1288 occurs. This happens because MySQL does not support updating or deleting data directly from a CTE (Common Table Expression) or a derived table that uses window functions like ROW_NUMBER(). MySQL restricts DELETE statements from targeting temporary result sets built with window logic. Below is the query:

```
-- WITH RowNumbered AS (
-- SELECT
-- 	Name,
-- 	ROW_NUMBER() OVER(PARTITION BY Name ORDER BY (SELECT NULL))
--     AS row_num
-- FROM health
-- )
-- DELETE FROM RowNumbered
-- WHERE row_num > 1;
```

With that, I decided to add a unique ID column and pkan to use self-join to keep the row with the same Name and delete any subsequent rows that share that same Name:
```
ALTER TABLE health ADD COLUMN id INT AUTO_INCREMENT PRIMARY KEY FIRST;
```

Temporarily disable safe update mode and self join to only select unique id of a line:
```
-- SET SQL_SAFE_UPDATES = 0; 
-- DELETE t1 FROM health t1
-- JOIN health t2 
--     ON t1.Name = t2.Name 
--     AND t1.id > t2.id;
```

However with this, I ran into another problem. This time, Error 2013 happens and it happens because the self-join query is taking too long and timing out. Without an index, MySQL has to compare every single row against every other row, which overwhelms the server memory and causes it to drop the connection. To fix this and make the deletion fast and safe, I used a Temporary Table approach. This avoids heavy self-joins entirely. However, Error 2013 happens again and I decided move on with deduplication. This issue seems to happen because the local server timeout limit was hit.
```
-- CREATE TEMPORARY TABLE keep_ids AS
-- SELECT MIN(id) AS id
-- FROM health
-- GROUP BY Name;
-- DELETE FROM health 
-- WHERE id NOT IN (SELECT id FROM keep_ids);
-- DROP TEMPORARY TABLE keep_ids;
```

I decided to move on and put duplicate check in Power BI only. This still intacts with our objective of using this as learning material for SQL.

## 3. Data Analysis

There are several correlation that I want to see and we will go through it each and have a insight and recommendation in the end.

### 3.1 Seeing the Correlation Between Age Group and Medical Condition

```
WITH CategorizedHealth AS (
    SELECT 
        *,
        CASE 
            WHEN Age BETWEEN 0 AND 17 THEN '0-17 (Pediatric)'
            WHEN Age BETWEEN 18 AND 30 THEN '18-30 (Young Adult)'
            WHEN Age BETWEEN 31 AND 50 THEN '31-50 (Adult)'
            WHEN Age BETWEEN 51 AND 65 THEN '51-65 (Middle-Aged)'
            ELSE '65+ (Senior)'
        END AS Age_Group
    FROM health
)
SELECT 
    Age_Group,
    `Medical Condition`,
    COUNT(*) AS Patients_With_Condition,
    SUM(COUNT(*)) OVER (PARTITION BY Age_Group) AS Total_Age_Group_Patients
FROM CategorizedHealth
GROUP BY Age_Group, `Medical Condition`
ORDER BY Age_Group, Patients_With_Condition DESC;
```

### 3.2. Seeing the Correlation Between Gender and Medical Condition
```
SELECT
    Gender,
    COALESCE(`Medical Condition`, '--- Total for this Gender ---') AS `Medical Condition`,
    COUNT(*) AS Total_Patients
FROM health
GROUP BY Gender, `Medical Condition` WITH ROLLUP;
```

### 3.3. Seeing the Correlation Between Blood Type and Medical Condition
```
SELECT
    `Blood Type`,
    `Medical Condition`,
    COUNT(*) AS Total_Patients,
    SUM(COUNT(*)) OVER (PARTITION BY `Blood Type`) AS Total_Blood_Type_Patients
FROM health
GROUP BY `Blood Type`, `Medical Condition`
ORDER BY `Blood Type`, Total_Patients DESC;
```

### 3.4. Seeing the Correlation Between Insurance Provider and Medical Condition
```
SELECT
    `Insurance Provider`,
    `Medical Condition`,
    COUNT(*) AS Total_Patients,
    SUM(COUNT(*)) OVER (PARTITION BY `Insurance Provider`) AS Total_Insurance_Provider_Patients
FROM health
GROUP BY `Insurance Provider`, `Medical Condition`
ORDER BY `Insurance Provider`, Total_Patients DESC;
```

### 3.5. Seeing the Sum of Billing Amount for Every Medical Condition
```
SELECT
    `Medical Condition`,
    ROUND(SUM(`Billing Amount`), 0) AS Total_Billing_Amount
FROM health
GROUP BY `Medical Condition` WITH ROLLUP;
```

### 3.6. Seeing the Billing Amount for every Insurance Provider
```
SELECT
    `Insurance Provider`,
    ROUND(SUM(`Billing Amount`), 0) AS Total_Billing_Amount
FROM health
GROUP BY `Insurance Provider` WITH ROLLUP;
```

### 3.7. Seeing the Length of Stay for every Medical Condition

```
SELECT
    AVG(DATEDIFF(`Discharge Date`, `Date of Admission`)) AS Length_of_stay,
    `Medical Condition`
FROM health
GROUP BY `Medical Condition`
ORDER BY Length_of_stay DESC;
```

## 4. Insight and Recommendation

This dataset on Kaggle was to serve as a material for learning, be it data science and data analysis enthusiasts. There are 54967 patients admitted in this dataset.

I first divide patients into group of five based on their age. There are Pediatric (0-17) Young Adult (18-30), Adult (31-50), Middle-Aged (51-65), and Senior (65+). We can see that adult and senior groups have somewhat similar patients with senior have only a bit more to adult, while both accounted for the most patients in the age group. We can also see that the numbers for each condition are somewhat similar, mostly because the data is synthetic. 

Next, I checked the correlation between gender and medical condition, which shows similar portion between male and female in the dataset. I used WITH ROLLUP function, so we are able to see the total patients of each gender. Again, the similarity is most probably because the data was synthesized for learning.

I went to see correlation between blood type and medical condition, to which I found they also show similar portion. With Rhesus negative seen as rare blood type, the shown data is highly unlikely.

Furthermore, I checked the correlation between insurance provider, their medical condition, and total patients in each group. Medicare shows up at the top of the list, beating the other four insurance provider with slight difference. There is no significant difference in every medical condition to their insurance provider. While so, the total billing amount for Diabetes is the highest among all the other medical conditions, while Cancer is the lowest. While Medicare has the most patients who claimed for their insurance, Blue Cross has the highest billing amount among the five providers.

I tried to see the length of stay for each medical condition but the difference is little, that it is negligible. The average length of stay is around 15 days. Lastly, I checked the amount of patients for every medical condition. The data shows Arthritis was the leading medical condition in the dataset.

I found a lot of these data to be highly synthesized that it might not be a good learning material. As a general learn for SQL yes, but probably would not be as useful to try to relate it to actual real-life dataset. The only part that I found interesting was the age group difference which can somehow shows relevance to real life. Most adult and senior groups are happen to have the most patients, followed by Middle-Aged. These medical conditions is generally known happened to adults nevertheless. For this, I believe making it lean on more real life situation is better for this dataset.
